# COPY INTO: Loading Data from Parquet Files

## Overview

This notebook demonstrates the **COPY INTO** command for loading data from Parquet files into Delta tables. COPY INTO is an idempotent, incremental data loading operation that's perfect for batch ingestion.

### What You'll Learn

* Inspecting Parquet file schemas before loading
* Loading data with type casting (Parquet types → String types)
* Loading data with matching schemas (proper type alignment)
* Idempotent loading (preventing duplicate loads)
* Schema evolution with `mergeSchema` option

### Key Concepts

**COPY INTO advantages:**
* **Idempotent**: Files are loaded only once by default
* **Incremental**: Only new files are processed
* **Type-safe**: Schema validation before loading
* **Performant**: Optimized for batch loads

**Schema Evolution:**
* `FORMAT_OPTIONS ('mergeSchema' = 'true')`: Infers and merges source schemas
* `COPY_OPTIONS ('mergeSchema' = 'true')`: Allows target table schema to evolve

### Prerequisites

* Unity Catalog enabled workspace
* Write access to `databricksmayank.bronze` catalog
* Parquet files in `/Volumes/databricksmayank/bronze/copyinto/raw/`

## Step 1: Inspect Source Parquet File

Before loading data, inspect the Parquet file to understand its schema and data types. This helps you decide whether you need type conversions or can load data directly.

**Query below:** Reads a sample Parquet file directly to examine its structure.

In [0]:
%sql
select * from parquet.`/Volumes/databricksmayank/bronze/copyinto/raw/part-00000-tid-1800520145580676501-2fb00642-a896-4599-9957-2c7260f04aa7-132-1.c000.snappy.parquet`

## Scenario 1: Loading with Type Casting

This scenario demonstrates loading Parquet data into a table with **different data types** (all STRING columns).

**Challenge:** The Parquet file has:
* `order_id`: INTEGER
* `customer_id`: INTEGER
* `order_date`: DATE
* `order_amount`: DOUBLE
* `order_status`: STRING

**Solution:** Use a SELECT statement with explicit CAST operations to convert all columns to STRING.

**Steps:**
1. Drop existing table (if any)
2. Create table with all STRING columns
3. Use COPY INTO with SELECT and CAST statements

In [0]:
%%sql
drop table if exists databricksmayank.bronze.copyintotbl

In [0]:
%sql
create table databricksmayank.bronze.copyintotbl (
  order_id string,
  customer_id string,
  order_date string,
  order_amount string,
  order_status string
)

In [0]:
%%sql
COPY INTO databricksmayank.bronze.copyintotbl
FROM (
  SELECT 
    CAST(order_id AS STRING) AS order_id,
    CAST(customer_id AS STRING) AS customer_id,
    CAST(order_date AS STRING) AS order_date,
    CAST(order_amount AS STRING) AS order_amount,
    order_status
  FROM '/Volumes/databricksmayank/bronze/copyinto/raw/'
)
FILEFORMAT = PARQUET

## Scenario 2: Loading with Matching Schema

This scenario demonstrates the **recommended approach**: creating a table with data types that match the Parquet source.

**Table schema matches Parquet:**
* `order_id`: INT
* `customer_id`: INT
* `order_date`: DATE
* `order_amount`: DOUBLE
* `order_status`: STRING

**Benefits:**
* No type conversion needed
* Better query performance
* Proper data type enforcement
* No explicit CAST statements required

**Schema evolution options enabled:**
* `FORMAT_OPTIONS ('mergeSchema' = 'True')`: Allows schema inference
* `COPY_OPTIONS ('mergeSchema' = 'True')`: Allows table schema evolution

In [0]:
%%sql
drop table if exists databricksmayank.bronze.copyintotbl

In [0]:
%sql
create table databricksmayank.bronze.copyintotbl (
  order_id int,
  customer_id INT,
  order_date DATE,
  order_amount DOUBLE,
  order_status string
)

In [0]:
%%sql
COPY INTO databricksmayank.bronze.copyintotbl
FROM '/Volumes/databricksmayank/bronze/copyinto/raw/'

FILEFORMAT = PARQUET
FORMAT_OPTIONS ("mergeSchema"="True")
COPY_OPTIONS ("mergeSchema"="True")

In [0]:
%sql
select * from databricksmayank.bronze.copyintotbl

## Demo: Idempotency Testing

This section demonstrates COPY INTO's **idempotent behavior**: files that have already been loaded are automatically skipped.

**Test setup:**
1. Upload another copy of the same Parquet file to the raw directory
2. Run COPY INTO again (cell below)

**Expected result:**
* No new rows are inserted
* `num_affected_rows = 0` (no files processed)
* COPY INTO automatically detects that files have been loaded before
* No duplicate data in the table

**Why this matters:** Idempotency ensures data quality and prevents duplicate records, making COPY INTO safe for retry scenarios and scheduled jobs.

In [0]:
%%sql
COPY INTO databricksmayank.bronze.copyintotbl
FROM '/Volumes/databricksmayank/bronze/copyinto/raw/'

FILEFORMAT = PARQUET
FORMAT_OPTIONS ("mergeSchema"="True")
COPY_OPTIONS ("mergeSchema"="True")

In [0]:
%%sql
select * from databricksmayank.bronze.copyintotbl

## Demo: Schema Evolution with New Columns

This section demonstrates **automatic schema evolution** when new columns appear in the source data.

**Test setup:**
1. Upload an evolved Parquet file with additional columns:
   * `discount`: DOUBLE (new column)
   * `payment_method`: STRING (new column)
2. Run COPY INTO again with `mergeSchema` enabled (cell below)

**Expected result:**
* Table schema automatically evolves to include new columns
* New columns are added: `discount` and `payment_method`
* Existing rows have NULL values for new columns
* New rows have actual values for new columns
* No pipeline failure despite schema mismatch

**How it works:**
* `FORMAT_OPTIONS ('mergeSchema' = 'True')`: Detects schema changes in source files
* `COPY_OPTIONS ('mergeSchema' = 'True')`: Applies schema changes to target table

**Compare to Auto Loader:** Auto Loader's rescue mode captures new columns in `_rescued_data` JSON. COPY INTO's `mergeSchema` directly adds new columns to the table schema.

In [0]:
%%sql
COPY INTO databricksmayank.bronze.copyintotbl
FROM '/Volumes/databricksmayank/bronze/copyinto/raw/'

FILEFORMAT = PARQUET
FORMAT_OPTIONS ("mergeSchema"="True")
COPY_OPTIONS ("mergeSchema"="True")

In [0]:
%%sql
select * from databricksmayank.bronze.copyintotbl

## COPY INTO Syntax Reference

### Basic Syntax

```sql
COPY INTO target_table
FROM 'source_path'
FILEFORMAT = format
[FORMAT_OPTIONS (option = value [, ...])]
[COPY_OPTIONS (option = value [, ...])]
```

### With SELECT and Type Conversion

```sql
COPY INTO target_table
FROM (
  SELECT 
    CAST(col1 AS type) AS col1,
    col2,
    expression AS col3
  FROM 'source_path'
)
FILEFORMAT = format
```

### File Formats

| Format | Example |
| --- | --- |
| PARQUET | `FILEFORMAT = PARQUET` |
| JSON | `FILEFORMAT = JSON` |
| CSV | `FILEFORMAT = CSV` |
| AVRO | `FILEFORMAT = AVRO` |
| ORC | `FILEFORMAT = ORC` |

### Common FORMAT_OPTIONS

| Option | Type | Description | Example |
| --- | --- | --- | --- |
| mergeSchema | boolean | Infer and merge schemas across files | `'mergeSchema' = 'true'` |
| inferSchema | boolean | Automatically infer data types | `'inferSchema' = 'true'` |
| header | boolean | CSV has header row | `'header' = 'true'` |
| delimiter | string | CSV delimiter character | `'delimiter' = ','` |
| dateFormat | string | Date parsing format | `'dateFormat' = 'yyyy-MM-dd'` |

### Common COPY_OPTIONS

| Option | Type | Description | Example |
| --- | --- | --- | --- |
| mergeSchema | boolean | Allow target schema to evolve | `'mergeSchema' = 'true'` |
| force | boolean | Reload files (disable idempotency) | `'force' = 'true'` |

### File Selection

**Load all files in directory:**
```sql
FROM '/path/to/files/'
```

**Load specific files:**
```sql
FROM '/path/to/files/'
FILES = ('file1.parquet', 'file2.parquet')
```

**Load with pattern matching:**
```sql
FROM '/path/to/files/'
PATTERN = '*.parquet'
```

### Validation

**Validate all data without loading:**
```sql
COPY INTO target_table
FROM 'source_path'
FILEFORMAT = PARQUET
VALIDATE ALL
```

**Validate first N rows:**
```sql
VALIDATE 100 ROWS
```

## Summary

Congratulations! You've successfully learned how to use COPY INTO for loading Parquet data into Delta tables.

### What You Accomplished

✅ **Inspected Parquet schema** to understand source data types  
✅ **Loaded with type casting** using SELECT and CAST statements  
✅ **Loaded with matching schema** for optimal performance  
✅ **Verified idempotency** by preventing duplicate loads  
✅ **Evolved table schema** automatically with new columns  

### Key Takeaways

1. **Always inspect source data first** - Understanding the schema helps you design the target table correctly
2. **Match data types when possible** - Avoid unnecessary type conversions for better performance
3. **Use explicit CAST for type mismatches** - Control type conversion with SELECT statements
4. **Idempotency is automatic** - Files are loaded only once by default
5. **Schema evolution requires mergeSchema** - Enable both FORMAT_OPTIONS and COPY_OPTIONS for automatic evolution

---

## Best Practices

### Schema Design

**✅ Do:**
* Match target table types to source file types
* Use appropriate data types (INT for IDs, DATE for dates, DOUBLE for decimals)
* Plan for schema evolution in advance

**❌ Don't:**
* Use STRING for everything (loses type safety and performance)
* Ignore schema mismatches (will cause load failures)

### Type Conversion

**When types don't match:**
```sql
COPY INTO target_table
FROM (
  SELECT 
    CAST(col1 AS target_type) AS col1,
    CAST(col2 AS target_type) AS col2
  FROM 'source_path'
)
FILEFORMAT = PARQUET
```

### Schema Evolution

**Enable mergeSchema for evolving schemas:**
```sql
COPY INTO target_table
FROM 'source_path'
FILEFORMAT = PARQUET
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
```

**When to use:**
* ✅ Controlled environments where new columns are expected
* ✅ Backward-compatible schema changes
* ❌ Production systems with strict schema requirements (use schema validation instead)

### Idempotency

**Default behavior (idempotent):**
* Files are loaded only once
* Safe for retries and scheduled jobs
* Prevents duplicate data

**Override with force option:**
```sql
COPY INTO target_table
FROM 'source_path'
FILEFORMAT = PARQUET
COPY_OPTIONS ('force' = 'true')  -- Reloads all files
```

**Use force only when:**
* Files have been modified and need reloading
* You intentionally want to reload data

### Error Handling

**Common errors:**

| Error | Cause | Solution |
| --- | --- | --- |
| DELTA_FAILED_TO_MERGE_FIELDS | Type mismatch | Use explicit CAST in SELECT |
| PARSE_SYNTAX_ERROR | Syntax issue | Check FORMAT_OPTIONS/COPY_OPTIONS syntax |
| TABLE_NOT_FOUND | Missing table | Create table first |
| Permission denied | Access issue | Check volume/catalog permissions |

### Performance Tips

1. **Partition your data** - Organize source files by date/region for faster queries
2. **Use appropriate file sizes** - Target 128MB-1GB per file
3. **Leverage VALIDATE** - Test loads before executing:
   ```sql
   COPY INTO target_table
   FROM 'source_path'
   FILEFORMAT = PARQUET
   VALIDATE ALL  -- Validates without writing
   ```
4. **Monitor loads** - Check `num_affected_rows` and `num_inserted_rows` in results

---

## COPY INTO vs Auto Loader

| Feature | COPY INTO | Auto Loader |
| --- | --- | --- |
| **Use case** | Batch loading | Streaming ingestion |
| **Execution** | Run on-demand or scheduled | Continuous stream |
| **Idempotency** | Built-in (file-based) | Checkpoint-based |
| **Schema evolution** | mergeSchema option | rescue/addNewColumns modes |
| **Best for** | Hourly/daily batch jobs | Near-real-time ingestion |
| **Syntax** | SQL COPY INTO | PySpark readStream |

**When to use COPY INTO:**
* Scheduled batch loads (hourly, daily)
* One-time or ad-hoc data migrations
* Simple file-to-table loading
* SQL-first workflows

**When to use Auto Loader:**
* Continuous file monitoring
* Near-real-time requirements
* Complex transformations during ingestion
* Need for checkpoint management

---

## Next Steps

**For Development:**
* Experiment with different file formats (JSON, CSV, Avro)
* Try pattern-based file selection with PATTERN option
* Use VALIDATE to test loads before executing
* Implement error handling and monitoring

**For Production:**
* Create Databricks Jobs for scheduled COPY INTO operations
* Set up alerting on load failures
* Implement data quality checks before loading
* Use Delta table constraints to enforce data quality
* Archive processed files after successful loads
* Monitor performance metrics (rows loaded, execution time)

**Additional Learning:**
* [COPY INTO Documentation](https://docs.databricks.com/sql/language-manual/delta-copy-into/)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)
* [Schema Evolution Guide](https://docs.databricks.com/delta/schema-evolution.html)

---

**Questions or Issues?**  
Refer to the [Databricks Community](https://community.databricks.com/) or [Documentation](https://docs.databricks.com/)